In [13]:
from quickdraw import QuickDrawDataGroup
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

CATEGORIES = ['whale', 'panda', 'tiger', 'raccoon', 'rabbit', 'cat', 'fish', 'flamingo', 'frog', 'rhinoceros', 'shark', 'dragon', 'sheep', 'hedgehog', 'bear']
SAMPLES = 500
IMAGE_SIZE = 64
BATCH_SIZE = 32
NUM_EPOCHS = 15

label_map = {category: idx for idx, category in enumerate(CATEGORIES)}

In [14]:
images = []
labels = []

for category in CATEGORIES:
    group = QuickDrawDataGroup(category, max_drawings=SAMPLES)
    for drawing in group.drawings:
        img = np.array(drawing.image.convert('L').resize((IMAGE_SIZE, IMAGE_SIZE)), dtype=np.float32)
        images.append(img)
        labels.append(label_map[category])

X = np.array(images, dtype=np.float32) / 255.0
y = np.array(labels, dtype=np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

loading whale drawings
load complete
loading panda drawings
load complete
loading tiger drawings
load complete
loading raccoon drawings
load complete
loading rabbit drawings
load complete
loading cat drawings
load complete
loading fish drawings
load complete
loading flamingo drawings
load complete
loading frog drawings
load complete
loading rhinoceros drawings
load complete
loading shark drawings
load complete
loading dragon drawings
load complete
loading sheep drawings
load complete
loading hedgehog drawings
load complete
loading bear drawings
load complete
Train: (6000, 64, 64), Test: (1500, 64, 64)


In [15]:
X_train_t = torch.tensor(X_train[:, None, :, :], dtype=torch.float32)
X_test_t = torch.tensor(X_test[:, None, :, :], dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE, shuffle=False)

In [16]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=len(CATEGORIES)):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * (IMAGE_SIZE // 4) * (IMAGE_SIZE // 4), 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [17]:
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb).argmax(dim=1)
            all_preds.append(preds.cpu())
            all_labels.append(yb.cpu())

    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()
    test_acc = accuracy_score(y_true, y_pred)
    train_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch: {epoch + 1} | loss: {train_loss:.4f} | acc: {test_acc:.4f}")

Epoch: 1 | loss: 2.6094 | acc: 0.2940
Epoch: 2 | loss: 2.1135 | acc: 0.3860
Epoch: 3 | loss: 1.8997 | acc: 0.4347
Epoch: 4 | loss: 1.7344 | acc: 0.4413
Epoch: 5 | loss: 1.6000 | acc: 0.4580
Epoch: 6 | loss: 1.4612 | acc: 0.4673
Epoch: 7 | loss: 1.2887 | acc: 0.5040
Epoch: 8 | loss: 1.1416 | acc: 0.5107
Epoch: 9 | loss: 0.9823 | acc: 0.5047
Epoch: 10 | loss: 0.8295 | acc: 0.5247
Epoch: 11 | loss: 0.6982 | acc: 0.5053
Epoch: 12 | loss: 0.5761 | acc: 0.4840
Epoch: 13 | loss: 0.4641 | acc: 0.5033
Epoch: 14 | loss: 0.3609 | acc: 0.4873
Epoch: 15 | loss: 0.2687 | acc: 0.5013


In [18]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        preds = model(xb).argmax(dim=1)
        all_preds.append(preds.cpu())
        all_labels.append(yb.cpu())

y_pred = torch.cat(all_preds).numpy()
y_true = torch.cat(all_labels).numpy()

print(classification_report(y_true, y_pred, target_names=CATEGORIES, zero_division=0))

              precision    recall  f1-score   support

       whale       0.42      0.48      0.45       100
       panda       0.58      0.43      0.49       100
       tiger       0.47      0.61      0.53       100
     raccoon       0.28      0.16      0.20       100
      rabbit       0.62      0.42      0.50       100
         cat       0.44      0.47      0.45       100
        fish       0.64      0.64      0.64       100
    flamingo       0.71      0.77      0.74       100
        frog       0.35      0.32      0.33       100
  rhinoceros       0.45      0.56      0.50       100
       shark       0.60      0.68      0.64       100
      dragon       0.47      0.27      0.34       100
       sheep       0.52      0.58      0.55       100
    hedgehog       0.68      0.69      0.68       100
        bear       0.32      0.44      0.37       100

    accuracy                           0.50      1500
   macro avg       0.50      0.50      0.49      1500
weighted avg       0.50   

In [19]:
from pathlib import Path
from PIL import Image

image_dir = Path('../../data')
image_files = sorted(image_dir.glob('*.png'))

for image_path in image_files:
    img = Image.open(image_path).convert('L').resize((IMAGE_SIZE, IMAGE_SIZE))
    img_array = np.array(img, dtype=np.float32).reshape(1, IMAGE_SIZE, IMAGE_SIZE) / 255.0
    img_tensor = torch.tensor(img_array, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        pred_idx = model(img_tensor.to(device)).argmax(dim=1).item()
    pred_label = CATEGORIES[pred_idx]
    print(f'{image_path.name}: predicted as {pred_label}')

bear.png: predicted as hedgehog
cat.png: predicted as raccoon
dragon.png: predicted as panda
fish.png: predicted as tiger
flamingo.png: predicted as rhinoceros
frog.png: predicted as hedgehog
hedgehog.png: predicted as panda
panda.png: predicted as bear
rabbit.png: predicted as raccoon
raccoon.png: predicted as panda
rhinoceros.png: predicted as raccoon
shark.png: predicted as raccoon
sheep.png: predicted as raccoon
tiger.png: predicted as raccoon
whale.png: predicted as raccoon
